# Liberia Ability Grouping — Descriptive Analysis

Exploratory descriptives to check for signal before re-doing the analysis.  
Data: `6_L_AG_full_wide_prepared.dta`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# paths
DATA_PATH = '/Users/mriduljoshi/Dropbox/Bridge TARL/AbilityGrouping/2_Data/2_Cleaned/Liberia/6_L_AG_full_wide_prepared.dta'

# read without converting value labels to categories (keeps numeric codes)
df_raw = pd.read_stata(DATA_PATH, convert_categoricals=False)
print(f'Raw dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print()
print('Column dtypes (selected):')
for col in ['treat', 'grade', 'std_grp', 'score_bl', 'score_ml', 'score_el', 'finsamp', 'strata']:
    if col in df_raw.columns:
        print(f'  {col}: {df_raw[col].dtype}, values: {sorted(df_raw[col].dropna().unique()[:8].tolist())}')

## 1. Sample overview

In [ ]:
# final sample
df = df_raw[df_raw['finsamp'] == 1].copy()
print(f'Final sample (finsamp==1): {len(df):,} students')
print()

# by treatment
print('--- By treatment ---')
print(df['treat'].value_counts().rename({0: 'Control', 1: 'Treatment'}).to_string())
print()

# by grade
print('--- By grade ---')
print(df.groupby(['grade', 'treat']).size().unstack(fill_value=0).rename(columns={0:'Control', 1:'Treatment'}).to_string())
print()

# by reading group
grp_labels = {1: 'Yellow (G1-2 low)', 2: 'Orange (G1-2 high)', 3: 'Blue (G3-4 low)', 4: 'Purple (G3-4 high)'}
print('--- By reading group ---')
print(df.groupby(['std_grp', 'treat']).size().unstack(fill_value=0).rename(columns={0:'Control', 1:'Treatment'}).to_string())
print()

# by strata
print('--- By randomization strata (acad_year x g12) ---')
print(df.groupby(['strata', 'treat']).size().unstack(fill_value=0).rename(columns={0:'Control', 1:'Treatment'}).to_string())

In [ ]:
# number of academies
print('--- Academies ---')
print(f"Total academies: {df['academycode'].nunique()}")
acad_treat = df.groupby('academycode')['treat'].mean()
print(f"Pure treatment academies (treat=1 for all): {(acad_treat == 1).sum()}")
print(f"Pure control academies (treat=0 for all): {(acad_treat == 0).sum()}")
print(f"Mixed academies: {((acad_treat > 0) & (acad_treat < 1)).sum()}")

## 2. Score availability / attrition

In [ ]:
score_vars = ['score_bl', 'score_ml', 'score_el']
score_labels = {'score_bl': 'Baseline', 'score_ml': 'Midline', 'score_el': 'Endline'}

print('=== Score availability in final sample ===')
for v in score_vars:
    n = df[v].notna().sum()
    pct = n / len(df) * 100
    print(f"{score_labels[v]}: {n:,} / {len(df):,} ({pct:.1f}%)")

print()
print('--- By treatment ---')
for treat_val, label in [(0, 'Control'), (1, 'Treatment')]:
    sub = df[df['treat'] == treat_val]
    print(f"{label} (N={len(sub):,}):")
    for v in score_vars:
        n = sub[v].notna().sum()
        pct = n / len(sub) * 100
        print(f"  {score_labels[v]}: {n:,} ({pct:.1f}%)")

print()
# differential attrition test
print('--- Differential attrition (endline missing): t-test treat vs control ---')
df['missing_el'] = df['score_el'].isna().astype(int)
t, p = stats.ttest_ind(df.loc[df['treat']==1, 'missing_el'], df.loc[df['treat']==0, 'missing_el'])
mean_t = df.loc[df['treat']==1, 'missing_el'].mean()
mean_c = df.loc[df['treat']==0, 'missing_el'].mean()
print(f"  Missing EL: Treatment={mean_t:.3f}, Control={mean_c:.3f}")
print(f"  t={t:.3f}, p={p:.3f}")

## 3. Descriptive statistics — test scores

In [ ]:
def desc_table(data, var, by='treat', label_map={0: 'Control', 1: 'Treatment'}):
    """Return mean, sd, p25, p75, N by group."""
    rows = []
    for val, lbl in label_map.items():
        sub = data.loc[data[by] == val, var].dropna()
        rows.append({
            'Group': lbl,
            'N': len(sub),
            'Mean': sub.mean(),
            'SD': sub.std(),
            'p25': sub.quantile(0.25),
            'Median': sub.median(),
            'p75': sub.quantile(0.75),
            'Min': sub.min(),
            'Max': sub.max(),
        })
    return pd.DataFrame(rows).set_index('Group').round(3)

for v, lbl in score_labels.items():
    print(f'\n=== {lbl} score ({v}) ===')
    print(desc_table(df, v).to_string())
    
    # by grade too
    for g in sorted(df['grade'].dropna().unique()):
        sub = df[df['grade'] == g]
        tbl = desc_table(sub, v)
        print(f"  Grade {int(g)}: Control mean={tbl.loc['Control','Mean']:.2f} (N={int(tbl.loc['Control','N'])}),  Treatment mean={tbl.loc['Treatment','Mean']:.2f} (N={int(tbl.loc['Treatment','N'])})")

## 4. Balance check (baseline scores by treatment)

In [ ]:
print('=== Balance: Baseline score by treatment ===')
print()

# overall
tval, pval = stats.ttest_ind(
    df.loc[df['treat']==1, 'score_bl'].dropna(),
    df.loc[df['treat']==0, 'score_bl'].dropna()
)
mean_t = df.loc[df['treat']==1, 'score_bl'].mean()
mean_c = df.loc[df['treat']==0, 'score_bl'].mean()
print(f"Overall — Control: {mean_c:.3f}, Treatment: {mean_t:.3f}, diff: {mean_t-mean_c:.3f}, t={tval:.3f}, p={pval:.3f}")
print()

# by grade
print('By grade:')
for g in sorted(df['grade'].dropna().unique()):
    sub = df[df['grade']==g]
    t_scores = sub.loc[sub['treat']==1, 'score_bl'].dropna()
    c_scores = sub.loc[sub['treat']==0, 'score_bl'].dropna()
    tval, pval = stats.ttest_ind(t_scores, c_scores)
    print(f"  Grade {int(g)}: Control={c_scores.mean():.3f} (N={len(c_scores)}), Treatment={t_scores.mean():.3f} (N={len(t_scores)}), diff={t_scores.mean()-c_scores.mean():.3f}, p={pval:.3f}")

# standardized scores
print()
print('Standardized baseline score:')
tval, pval = stats.ttest_ind(
    df.loc[df['treat']==1, 'std_score_bl'].dropna(),
    df.loc[df['treat']==0, 'std_score_bl'].dropna()
)
mean_t = df.loc[df['treat']==1, 'std_score_bl'].mean()
mean_c = df.loc[df['treat']==0, 'std_score_bl'].mean()
print(f"  Control: {mean_c:.4f}, Treatment: {mean_t:.4f}, diff: {mean_t-mean_c:.4f}, p={pval:.3f}")

## 5. Treatment effects — raw and by subgroup

In [ ]:
from scipy.stats import ttest_ind

def treatment_effect(data, outcome, group_var=None, group_val=None, label=''):
    """Compute simple treatment effect (difference in means + t-test)."""
    if group_var is not None:
        sub = data[data[group_var] == group_val]
    else:
        sub = data
    
    t = sub.loc[sub['treat']==1, outcome].dropna()
    c = sub.loc[sub['treat']==0, outcome].dropna()
    
    if len(t) < 5 or len(c) < 5:
        return None
    
    diff = t.mean() - c.mean()
    se = np.sqrt(t.var()/len(t) + c.var()/len(c))
    tstat, pval = ttest_ind(t, c)
    c_sd = c.std()
    std_eff = diff / c_sd if c_sd > 0 else np.nan
    
    return {
        'Label': label,
        'N_treat': len(t), 'N_ctrl': len(c),
        'Mean_ctrl': c.mean(), 'Mean_treat': t.mean(),
        'Diff': diff, 'SE': se,
        'Std_eff': std_eff,
        't': tstat, 'p': pval
    }


outcomes = [
    ('std_score_el', 'Endline (std)'),
    ('std_score_ml', 'Midline (std)'),
    ('score_el', 'Endline (raw)'),
]

for outcome, out_label in outcomes:
    if df[outcome].notna().sum() < 10:
        continue
    print(f'\n=== Treatment effects: {out_label} ===')
    rows = []
    
    # overall
    r = treatment_effect(df, outcome, label='Overall')
    if r: rows.append(r)
    
    # by grade
    for g in sorted(df['grade'].dropna().unique()):
        r = treatment_effect(df, outcome, 'grade', g, label=f'Grade {int(g)}')
        if r: rows.append(r)
    
    # by reading group
    for grp in sorted(df['std_grp'].dropna().unique()):
        grp_name = grp_labels.get(int(grp), f'Grp {int(grp)}')
        r = treatment_effect(df, outcome, 'std_grp', grp, label=grp_name)
        if r: rows.append(r)
    
    result_df = pd.DataFrame(rows).set_index('Label')
    result_df = result_df[['N_ctrl', 'N_treat', 'Mean_ctrl', 'Mean_treat', 'Diff', 'SE', 'Std_eff', 'p']]
    print(result_df.round(3).to_string())

## 6. Score gains: BL → EL

In [ ]:
df['gain_raw'] = df['score_el'] - df['score_bl']
df['gain_std'] = df['std_score_el'] - df['std_score_bl']

print('=== Score gains (endline - baseline) ===')
print()

for gain_var, lbl in [('gain_raw', 'Raw gain'), ('gain_std', 'Std gain')]:
    print(f'--- {lbl} ---')
    r = treatment_effect(df, gain_var, label='Overall')
    if r:
        print(f"  Overall: ctrl={r['Mean_ctrl']:.3f}, treat={r['Mean_treat']:.3f}, diff={r['Diff']:.3f} (SE={r['SE']:.3f}, p={r['p']:.3f})")
    
    for g in sorted(df['grade'].dropna().unique()):
        r = treatment_effect(df, gain_var, 'grade', g)
        if r:
            print(f"  Grade {int(g)}: ctrl={r['Mean_ctrl']:.3f}, treat={r['Mean_treat']:.3f}, diff={r['Diff']:.3f} (p={r['p']:.3f})")
    print()

## 7. Figures: Score distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (var, lbl) in zip(axes, [('score_bl', 'Baseline'), ('score_ml', 'Midline'), ('score_el', 'Endline')]):
    for treat_val, color, label in [(0, '#2166ac', 'Control'), (1, '#d6604d', 'Treatment')]:
        sub = df.loc[df['treat']==treat_val, var].dropna()
        ax.hist(sub, bins=30, alpha=0.5, color=color, label=f'{label} (N={len(sub):,})', density=True)
    ax.set_title(f'{lbl} score')
    ax.set_xlabel('Score')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Score distributions: Treatment vs Control', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# By grade
grades = sorted(df['grade'].dropna().unique())
fig, axes = plt.subplots(len(grades), 3, figsize=(16, 4*len(grades)))

for i, g in enumerate(grades):
    sub_g = df[df['grade'] == g]
    for j, (var, lbl) in enumerate([('score_bl', 'Baseline'), ('score_ml', 'Midline'), ('score_el', 'Endline')]):
        ax = axes[i][j]
        for treat_val, color, label in [(0, '#2166ac', 'Control'), (1, '#d6604d', 'Treatment')]:
            sub = sub_g.loc[sub_g['treat']==treat_val, var].dropna()
            if len(sub) > 0:
                ax.hist(sub, bins=25, alpha=0.5, color=color, label=f'{label} (N={len(sub):,})', density=True)
        ax.set_title(f'Grade {int(g)} — {lbl}')
        ax.set_xlabel('Score')
        ax.legend(fontsize=7)

plt.suptitle('Score distributions by grade', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## 8. Figures: Treatment effects by grade and reading group

In [ ]:
# Forest-style plot: treatment effects on std endline, by grade and reading group
outcome = 'std_score_el'

labels, diffs, ses = [], [], []

# overall
r = treatment_effect(df, outcome, label='Overall')
if r: labels.append(r['Label']); diffs.append(r['Diff']); ses.append(r['SE'])

labels.append('')
diffs.append(np.nan)
ses.append(np.nan)

# by grade
for g in sorted(df['grade'].dropna().unique()):
    r = treatment_effect(df, outcome, 'grade', g, label=f'Grade {int(g)}')
    if r: labels.append(r['Label']); diffs.append(r['Diff']); ses.append(r['SE'])

labels.append('')
diffs.append(np.nan)
ses.append(np.nan)

# by reading group
for grp in sorted(df['std_grp'].dropna().unique()):
    grp_name = grp_labels.get(int(grp), f'Grp {int(grp)}')
    r = treatment_effect(df, outcome, 'std_grp', grp, label=grp_name)
    if r: labels.append(r['Label']); diffs.append(r['Diff']); ses.append(r['SE'])

y = np.arange(len(labels))
diffs = np.array(diffs, dtype=float)
ses = np.array(ses, dtype=float)

fig, ax = plt.subplots(figsize=(8, 0.5*len(labels)+1))
ax.errorbar(diffs, y, xerr=1.96*ses, fmt='o', color='#333333', ecolor='#888888',
            capsize=3, markersize=5)
ax.axvline(0, color='red', linestyle='--', alpha=0.5)
ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.set_xlabel('Treatment effect (std endline score, 95% CI)')
ax.set_title('Treatment effects on standardized endline score')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Baseline score distributions by reading group (check cutoff)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# grades 1-2
ax = axes[0]
for grp, color, lbl in [(1, '#f4d03f', 'Yellow (low)'), (2, '#e67e22', 'Orange (high)')]:
    sub = df.loc[(df['std_grp']==grp), 'score_bl'].dropna()
    ax.hist(sub, bins=20, alpha=0.6, color=color, label=f'{lbl} (N={len(sub):,})', density=True)
ax.axvline(23, color='red', linestyle='--', linewidth=2, label='Cutoff (23)')
ax.set_title('Grades 1-2: BL score by reading group')
ax.set_xlabel('Baseline score')
ax.legend()

# grades 3-4
ax = axes[1]
for grp, color, lbl in [(3, '#5dade2', 'Blue (low)'), (4, '#8e44ad', 'Purple (high)')]:
    sub = df.loc[(df['std_grp']==grp), 'score_bl'].dropna()
    ax.hist(sub, bins=20, alpha=0.6, color=color, label=f'{lbl} (N={len(sub):,})', density=True)
ax.axvline(14, color='red', linestyle='--', linewidth=2, label='Cutoff (14)')
ax.set_title('Grades 3-4: BL score by reading group')
ax.set_xlabel('Baseline score')
ax.legend()

plt.tight_layout()
plt.show()

## 10. BL → EL score correlation and gains

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# scatter: BL vs EL, by treatment
ax = axes[0]
for treat_val, color, label in [(0, '#2166ac', 'Control'), (1, '#d6604d', 'Treatment')]:
    sub = df[df['treat']==treat_val][['score_bl', 'score_el']].dropna()
    ax.scatter(sub['score_bl'], sub['score_el'], alpha=0.05, s=5, color=color, label=label)
    # regression line
    if len(sub) > 10:
        slope, intercept, r, _, _ = stats.linregress(sub['score_bl'], sub['score_el'])
        x_line = np.linspace(sub['score_bl'].min(), sub['score_bl'].max(), 100)
        ax.plot(x_line, slope*x_line+intercept, color=color, linewidth=2, label=f'{label} (r={r:.2f})')
ax.set_xlabel('Baseline score')
ax.set_ylabel('Endline score')
ax.set_title('BL → EL: scatter by treatment')
ax.legend()

# gain distribution
ax = axes[1]
for treat_val, color, label in [(0, '#2166ac', 'Control'), (1, '#d6604d', 'Treatment')]:
    sub = df.loc[df['treat']==treat_val, 'gain_raw'].dropna()
    ax.hist(sub, bins=30, alpha=0.5, color=color, label=f'{label} (mean={sub.mean():.1f})', density=True)
ax.axvline(0, color='black', linestyle='--', alpha=0.5)
ax.set_xlabel('Score gain (EL - BL)')
ax.set_ylabel('Density')
ax.set_title('Distribution of score gains')
ax.legend()

plt.tight_layout()
plt.show()

## 11. Academy-level variation (check randomization)

In [ ]:
# academy-level means
acad = df.groupby(['academycode', 'treat']).agg(
    n=('studyid', 'count'),
    mean_bl=('score_bl', 'mean'),
    mean_el=('score_el', 'mean'),
    mean_gain=('gain_raw', 'mean'),
).reset_index()

print('Academy-level summary:')
print(acad.groupby('treat')[['mean_bl', 'mean_el', 'mean_gain']].describe().round(2).to_string())

# plot academy-level mean gains
fig, ax = plt.subplots(figsize=(10, 4))
for treat_val, color, label in [(0, '#2166ac', 'Control'), (1, '#d6604d', 'Treatment')]:
    sub = acad[acad['treat']==treat_val]['mean_gain'].dropna()
    ax.hist(sub, bins=20, alpha=0.6, color=color, label=f'{label} (N={len(sub)} academies, mean={sub.mean():.2f})', density=True)
ax.set_xlabel('Academy mean score gain (EL - BL)')
ax.set_title('Distribution of academy-level mean gains')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# academy-level: BL vs gain, by treatment
fig, ax = plt.subplots(figsize=(8, 5))
for treat_val, color, label in [(0, '#2166ac', 'Control'), (1, '#d6604d', 'Treatment')]:
    sub = acad[acad['treat']==treat_val][['mean_bl', 'mean_gain']].dropna()
    ax.scatter(sub['mean_bl'], sub['mean_gain'], alpha=0.6, s=40, color=color, label=label)
ax.set_xlabel('Academy mean baseline score')
ax.set_ylabel('Academy mean score gain')
ax.set_title('Academy: BL score vs gain — treatment vs control')
ax.legend()
ax.axhline(0, color='black', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Within-class dispersion: does ability grouping reduce dispersion?

In [ ]:
# SD of scores within reading group vs within grade
# Treatment: within std_grp (ability group)
# Control: within grade

# student-level: SD of class peers
# approximate: compute SD within their actual grouping
df_t = df[df['treat']==1].copy()
df_c = df[df['treat']==0].copy()

# treatment: SD within ability group within academy
df_t['grp_sd_bl'] = df_t.groupby(['academycode', 'std_grp'])['score_bl'].transform('std')
df_t['grp_sd_el'] = df_t.groupby(['academycode', 'std_grp'])['score_el'].transform('std')

# control: SD within grade within academy
df_c['grp_sd_bl'] = df_c.groupby(['academycode', 'grade'])['score_bl'].transform('std')
df_c['grp_sd_el'] = df_c.groupby(['academycode', 'grade'])['score_el'].transform('std')

print('=== Within-group SD of scores ===')
print()
for var, lbl in [('grp_sd_bl', 'Baseline'), ('grp_sd_el', 'Endline')]:
    c_mean = df_c[var].mean()
    t_mean = df_t[var].mean()
    tval, pval = ttest_ind(df_t[var].dropna(), df_c[var].dropna())
    print(f"{lbl}: Control SD={c_mean:.3f}, Treatment SD={t_mean:.3f}, diff={t_mean-c_mean:.3f}, p={pval:.3f}")

## 13. Summary of signal

In [ ]:
print('=' * 60)
print('SIGNAL SUMMARY')
print('=' * 60)

# Key treatment effects
outcomes_to_check = [
    ('std_score_el', 'Endline score (std)'),
    ('std_score_ml', 'Midline score (std)'),
    ('gain_raw', 'Score gain (raw)'),
    ('gain_std', 'Score gain (std)'),
]

for outcome, lbl in outcomes_to_check:
    if df[outcome].notna().sum() < 10:
        print(f'{lbl}: insufficient data')
        continue
    r = treatment_effect(df, outcome)
    if r:
        sig = '***' if r['p'] < 0.01 else ('**' if r['p'] < 0.05 else ('*' if r['p'] < 0.10 else ''))
        print(f"{lbl}: Diff={r['Diff']:+.3f} SD={r['Std_eff']:+.3f}σ, p={r['p']:.3f} {sig}")

print()
print('(Note: SE not clustered at academy level here — use Stata for inference)')